# Unit 2 — Run a policy yourself

Now write the code that passes observations into an existing ACT policy and sends its predicted actions to the simulator. The image, state, and action adapters are supplied. You will implement reset, action selection, and the control loop.

**Time:** four short exercises, about 15 minutes each; then one bounded ACT rollout. The CPU checks use a deterministic toy environment. The optional local run uses a saved ACT checkpoint and one development state.

The environment returns an observation and info on reset. Each step returns the next observation, reward, `terminated`, `truncated`, and info. `terminated` means the task ended. `truncated` means the time limit ended.

In [ ]:
from pathlib import Path
import os
import sys


def find_workspace(start=None):
    current = Path(start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent
    for folder in (current, *current.parents):
        if (folder / "manifests" / "act_references.yaml").is_file():
            return folder
    raise FileNotFoundError("Open this notebook inside the VLA course workspace.")

WORKSPACE = find_workspace()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Use local caches and prevent model or data downloads when importing libraries.
HF_CACHE = WORKSPACE / ".local-data" / "hf"
(HF_CACHE / "hub").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Workspace:", WORKSPACE)

import numpy as np
import torch
from vla_lab.teaching.runtime import RolloutResult


## Exercise 1 — Reset both parts

**Purpose:** start every trial from a known state and clear actions left in the policy queue.

**Contract:** `reset_episode(environment, policy, seed=None) -> observation`. `environment.reset(seed=seed)` returns `(observation, info)`. Call `policy.reset()` once after the environment reset. Return only the observation.

**Method:** reset the environment first. Then clear policy state so no action from the previous episode can run.

Pointers: Follow the Gymnasium reset result. Seed the reset when a seed is provided.

<details>
<summary>Detailed hint</summary>

Unpack `(observation, info)`, discard `info`, then call `policy.reset()`.
</details>

In [ ]:
def reset_episode(environment, policy, seed=None):
    """Reset the environment and policy; return the first observation."""
    raise NotImplementedError("Exercise 1: implement reset_episode")

In [ ]:
class TinyEnvironment:
    def __init__(self): self.seeds = []
    def reset(self, seed=None):
        self.seeds.append(seed)
        return {"step": 0}, {"seed": seed}

class TinyPolicy:
    def __init__(self): self.reset_count = 0
    def reset(self): self.reset_count += 1

toy_env, toy_policy = TinyEnvironment(), TinyPolicy()
first_observation = reset_episode(toy_env, toy_policy, seed=12)
assert first_observation == {"step": 0}, "Return the observation from reset."
assert toy_env.seeds == [12] and toy_policy.reset_count == 1, "Reset the environment and clear the policy once."
print("Reset check passed.")

## Exercise 2 — Select one action

**Purpose:** choose one action from a predicted action chunk.

**Contract:** input has shape `[1, T, 7]`: batch size 1, `T` time steps, and seven controller values. `offset` starts at zero. Return a vector with shape `[7]`, and keep the input type.

**Method:** remove the batch axis and select the requested time step. Raise a useful error for an invalid shape or offset.

Pointers: Check the batch axis, then check the time index.

<details>
<summary>Detailed hint</summary>

Use `chunk[0, offset]`. NumPy arrays and PyTorch tensors both support this indexing.
</details>

In [ ]:
def action_at(chunk, offset):
    """Return one seven-value action from a [1,T,7] chunk."""
    raise NotImplementedError("Exercise 2: implement action_at")

In [ ]:
test_chunk = torch.arange(21, dtype=torch.float32).reshape(1, 3, 7)
selected = action_at(test_chunk, 2)
assert selected.shape == (7,) and torch.equal(selected, test_chunk[0, 2]), "Select row 2 as a [7] action."
print("Action selection check passed.")

## Exercise 3 — Ask ACT for a chunk

**Purpose:** get the full predicted chunk so the control loop can choose how many actions to execute.

**Contract:** `predict_act_chunk(policy, batch) -> Tensor [1,T,7]`. The supplied processor has already prepared a batch with camera and state tensors. Use inference mode so this call does not store gradients.

**Method:** call ACT's `predict_action_chunk`. This method returns normalized action predictions. The supplied postprocessor converts one normalized action to controller units.

Pointers: Check the method name in the ACT policy. Make the prediction under `torch.no_grad()`.

<details>
<summary>Detailed hint</summary>

Return `policy.predict_action_chunk(batch)` from inside `with torch.no_grad():`.
</details>

In [ ]:
def predict_act_chunk(policy, batch):
    """Return ACT predictions with shape [1,T,7], without gradients."""
    raise NotImplementedError("Exercise 3: implement predict_act_chunk")

In [ ]:
class TinyChunkPolicy(TinyPolicy):
    def __init__(self):
        super().__init__()
        self.action_chunk = torch.nn.Parameter(torch.zeros((1, 3, 7)))
    def predict_action_chunk(self, batch):
        return self.action_chunk

predicted = predict_act_chunk(TinyChunkPolicy(), {})
assert predicted.shape == (1, 3, 7) and not predicted.requires_grad, "Return a chunk without a gradient graph."
print("Prediction check passed.")

## Exercise 4 — Write the control loop

**Purpose:** connect reset, observation preparation, policy prediction, and environment actions.

**Contract:** `run_policy_loop(environment, policy, prepare_observation, predict_chunk, action_to_controller, *, max_steps, execute_steps, seed=None) -> RolloutResult`. Predictions have shape `[1,T,7]`. Prepare and postprocess callbacks are supplied. Clip finite controller actions to `[-1,1]`. Stop on success, termination, truncation, or `max_steps`. Predict a new chunk after `execute_steps` actions.

**Method:** reset first. At each chunk boundary, prepare the current observation and predict. Execute up to the smaller of `execute_steps`, the chunk length, and the remaining step budget. Check `info["is_success"]`, `terminated`, and `truncated` after each step.

Pointers: Count policy calls separately from environment steps. Keep every attempted action in the result. Reject wrong action shapes and non-finite values.

<details>
<summary>Detailed hint</summary>

Use an outer loop for prediction calls and an inner loop for actions from one chunk. When the inner loop ends because the environment ended, break the outer loop too. `terminated` and `truncated` are different outcomes.
</details>

In [ ]:
def run_policy_loop(environment, policy, prepare_observation, predict_chunk,
                    action_to_controller, *, max_steps, execute_steps, seed=None):
    """Run bounded chunks; return RolloutResult with outcome and action counts."""
    raise NotImplementedError("Exercise 4: implement run_policy_loop")

In [ ]:
class TinyStepEnvironment(TinyEnvironment):
    def __init__(self, end_after=4, end_kind="timeout"):
        super().__init__(); self.end_after=end_after; self.end_kind=end_kind; self.step_count=0; self.actions=[]
    def reset(self, seed=None):
        self.step_count=0
        return super().reset(seed)
    def step(self, action):
        self.step_count += 1; self.actions.append(np.asarray(action).copy())
        done = self.step_count == self.end_after
        info = {"is_success": done and self.end_kind == "success"}
        return {"step": self.step_count}, 0.0, done and self.end_kind == "terminated", done and self.end_kind == "truncated", info

class TinyChunkPolicy(TinyPolicy):
    def __init__(self): super().__init__(); self.calls=0
    def predict_action_chunk(self, batch):
        self.calls += 1
        return torch.full((1, 3, 7), 1.2)

toy_env, toy_policy = TinyStepEnvironment(end_after=4), TinyChunkPolicy()
result = run_policy_loop(toy_env, toy_policy, lambda obs: obs, predict_act_chunk,
                         lambda action: action, max_steps=4, execute_steps=2, seed=4)
assert result.steps == 4 and result.policy_calls == 2, "Two chunks should run four environment steps."
assert result.clipped_values == 28 and all((action == 1.0).all() for action in toy_env.actions), "Clip each controller value to 1."
print("Control loop check passed:", result.termination_reason, result.steps, "steps")
for end_kind, expected_reason in (("success", "success"), ("terminated", "terminated"), ("truncated", "truncated")):
    ending_env = TinyStepEnvironment(end_after=2, end_kind=end_kind)
    ending_policy = TinyChunkPolicy()
    ended = run_policy_loop(ending_env, ending_policy, lambda obs: obs, predict_act_chunk,
                            lambda action: action, max_steps=8, execute_steps=3)
    assert ended.steps == 2 and ended.termination_reason == expected_reason, "Stop at success, termination, or truncation."

## Use your functions in LIBERO

This cell uses your loop with the supplied observation and action adapters. It uses ACT on one development state. It writes one video to `.local-data/course-runs/unit02`. This run is optional and does not use final test states.

In [ ]:
RUN_REAL_ROLLOUT = False
RUN_CHUNK_EXPERIMENT = False
REAL_STATE_ID = 0
REAL_MAX_STEPS = 100
chunk_trials = []
chunk_results_path = None
if RUN_REAL_ROLLOUT or RUN_CHUNK_EXPERIMENT:
    from vla_lab.policy.bundle import load_act_bundle
    from vla_lab.sim.episode import open_libero_task
    from vla_lab.teaching.runtime import prepare_libero_observation
    from uuid import uuid4

    assert 0 <= REAL_STATE_ID <= 9, "Use an exploration state from 0 to 9."
    checkpoint = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200"
    policy = preprocessor = postprocessor = None
    policy_load_error = None
    try:
        policy, preprocessor, postprocessor, manifest = load_act_bundle(checkpoint)
        policy.eval()
    except Exception as error:
        policy_load_error = f"{type(error).__name__}: {error}"

    def prepare_act_observation(observation):
        return prepare_libero_observation(observation, preprocessor)

    def act_controller_action(action):
        normalized = action.unsqueeze(0) if isinstance(action, torch.Tensor) else action[None, :]
        return postprocessor(normalized)[0]

    if RUN_REAL_ROLLOUT:
        if policy_load_error is not None:
            print("Real rollout skipped because policy loading failed:", policy_load_error)
        else:
            environment = None
            try:
                environment = open_libero_task("bowl_plate", REAL_STATE_ID, WORKSPACE / "configs", WORKSPACE / ".local-data")
                real_result = run_policy_loop(
                    environment, policy, prepare_act_observation, predict_act_chunk,
                    act_controller_action, max_steps=REAL_MAX_STEPS, execute_steps=10,
                    seed=20260930 + REAL_STATE_ID,
                )
                print(real_result)
                print("Policy calls:", real_result.policy_calls, "Environment steps:", real_result.steps)
            finally:
                if environment is not None:
                    environment.close()

    if RUN_CHUNK_EXPERIMENT:
        output = WORKSPACE / ".local-data/course-runs/unit02" / f"chunk-sweep-{uuid4().hex[:8]}"
        output.mkdir(parents=True, exist_ok=False)
        chunk_results_path = output / "results.json"
        for chunk_steps in (1, 5, 10):
            row = {"execute_steps": chunk_steps, "status": "failed",
                   "state": REAL_STATE_ID, "seed": 20260930 + REAL_STATE_ID}
            environment = None
            phase = "policy_load"
            try:
                if policy_load_error is not None:
                    raise RuntimeError(policy_load_error)
                phase = "environment_open"
                environment = open_libero_task("bowl_plate", REAL_STATE_ID, WORKSPACE / "configs", WORKSPACE / ".local-data")
                phase = "episode_run"
                trial = run_policy_loop(
                    environment, policy, prepare_act_observation, predict_act_chunk,
                    act_controller_action, max_steps=REAL_MAX_STEPS, execute_steps=chunk_steps,
                    seed=20260930 + REAL_STATE_ID,
                )
                trace = np.asarray(trial.actions, dtype=np.float32)
                row.update({"success": trial.success, "steps": trial.steps,
                            "policy_calls": trial.policy_calls,
                            "max_action_change": float(np.abs(np.diff(trace, axis=0)).max()) if len(trace) > 1 else 0.0,
                            "termination": trial.termination_reason, "status": "complete"})
            except Exception as error:
                row.update({"error_stage": phase, "error": f"{type(error).__name__}: {error}"})
            finally:
                if environment is not None:
                    try:
                        environment.close()
                    except Exception as error:
                        row["close_error"] = f"{type(error).__name__}: {error}"
                        row["status"] = "failed"
                chunk_trials.append(row)
                chunk_results_path.write_text(json.dumps(chunk_trials, indent=2) + "\n")
            print(row)
            print("Saved progress to", chunk_results_path)
        print("Completed or failed attempts:", len(chunk_trials))
else:
    print("Real runs are off. The checks above use a deterministic CPU environment.")

## Experiment — Change the chunk execution length

Set `RUN_CHUNK_EXPERIMENT = True` to run one bounded sweep. It compares `execute_steps` values 1, 5, and 10 on the same development state. Each trial resets the same policy and simulator with the same seed. It records success, environment steps, policy calls, maximum change between executed actions, and the stop reason. The file is saved under `.local-data/course-runs/unit02`. Keep failed trials. A different result does not prove that chunk length caused the change; small run-to-run differences can remain unclear.

In [ ]:
# Keep the rows from the live experiment cell; do not reset the results here.
from IPython.display import display
if chunk_trials:
    display(chunk_trials)
    if chunk_results_path is not None:
        print("Saved results:", chunk_results_path)
else:
    print("No real chunk trials have run in this kernel.")